In [5]:
# ==============================================================
# DATA PREP (FINAL):
# Sales + Orders + Stocks + Calendar + Lags + FX(EUR/TRY, USD/TRY via cross)
# ==============================================================

import pandas as pd
import numpy as np
import io
import requests

START_PERIOD = "2019-01"
END_PERIOD   = "2025-08"  # isterseniz değiştirin

# ------------------ Helpers ------------------
def month_to_season(m: int) -> str:
    if m in (12, 1, 2):
        return "Winter"
    if m in (3, 4, 5):
        return "Spring"
    if m in (6, 7, 8):
        return "Summer"
    return "Fall"

def fetch_ecb_sdmx_csv(series_code: str, start_period: str, end_period: str) -> pd.DataFrame:
    """
    ECB EXR veri setinden SDMX-CSV indirir.
    Örn:
      - EUR/TRY (TRY per EUR): "M.TRY.EUR.SP00.A"
      - EUR/USD (USD per EUR): "M.USD.EUR.SP00.A"
    """
    url = (
        f"https://sdw-wsrest.ecb.europa.eu/service/data/EXR/"
        f"{series_code}?startPeriod={start_period}&endPeriod={end_period}&format=csvdata"
    )
    r = requests.get(url, timeout=30)
    r.raise_for_status()
    df = pd.read_csv(io.StringIO(r.text))
    # Beklenen kolonlar: TIME_PERIOD, OBS_VALUE, ...
    df = df.rename(columns={"TIME_PERIOD": "ds", "OBS_VALUE": "value"})
    df["ds"] = pd.to_datetime(df["ds"].astype(str) + "-01")  # ay başı
    return df[["ds","value"]].sort_values("ds").reset_index(drop=True)

def safe_merge(left: pd.DataFrame, right: pd.DataFrame, on="ds") -> pd.DataFrame:
    return left.merge(right, on=on, how="left")

# ------------------ 1) Satış verisi ------------------
sales_data = {
"2019-01":22,"2019-02":60,"2019-03":60,"2019-04":45,"2019-05":127,
"2019-06":48,"2019-07":88,"2019-08":96,"2019-09":103,"2019-10":160,
"2019-11":181,"2019-12":66,"2020-01":84,"2020-02":124,"2020-03":204,
"2020-04":33,"2020-05":164,"2020-06":425,"2020-07":160,"2020-08":601,
"2020-09":68,"2020-10":184,"2020-11":88,"2020-12":32,"2021-01":205,
"2021-02":173,"2021-03":200,"2021-04":147,"2021-05":380,"2021-06":608,
"2021-07":112,"2021-08":66,"2021-09":41,"2021-10":482,"2021-11":192,
"2021-12":75,"2022-01":9,"2022-02":93,"2022-03":84,"2022-04":55,
"2022-05":160,"2022-06":220,"2022-07":49,"2022-08":33,"2022-09":182,
"2022-10":18,"2022-11":25,"2022-12":130,"2023-01":160,"2023-02":128,
"2023-03":76,"2023-04":104,"2023-05":128,"2023-06":388,"2023-07":264,
"2023-08":139,"2023-09":217,"2023-10":200,"2023-11":220,"2023-12":80,
"2024-01":192,"2024-02":124,"2024-03":284,"2024-04":292,"2024-05":164,
"2024-06":192,"2024-07":400,"2024-08":292,"2024-09":244,"2024-10":88,
"2024-11":124,"2024-12":56,"2025-01":160,"2025-02":96,"2025-03":218,
"2025-04":219,"2025-05":112,"2025-06":244,"2025-07":84
}
sales = pd.Series(sales_data, name="y")
sales.index = pd.to_datetime(sales.index)  # ay başı
df_sales = sales.rename_axis("ds").reset_index()

# ------------------ 2) Sipariş verisi ------------------
orders_txt = """SalesYear SalesMonth TotalQty
2019 1 16
2019 2 64
2019 3 72
2019 4 37
2019 5 135
2019 6 48
2019 7 96
2019 8 132
2019 9 137
2019 10 136
2019 11 173
2019 12 67
2020 1 64
2020 2 624
2020 3 197
2020 4 33
2020 5 164
2020 6 617
2020 7 132
2020 8 636
2020 9 56
2020 10 224
2020 11 48
2020 12 32
2021 1 249
2021 2 233
2021 3 368
2021 4 447
2021 5 596
2021 6 360
2021 7 144
2021 8 61
2021 9 378
2021 10 454
2021 11 228
2021 12 79
2022 1 6
2022 2 117
2022 3 92
2022 4 70
2022 5 240
2022 6 292
2022 7 57
2022 8 36
2022 9 222
2022 10 20
2022 11 40
2022 12 134
2023 1 229
2023 2 112
2023 3 80
2023 4 113
2023 5 160
2023 6 480
2023 7 252
2023 8 219
2023 9 173
2023 10 408
2023 11 196
2023 12 136
2024 1 284
2024 2 404
2024 3 324
2024 4 288
2024 5 158
2024 6 216
2024 7 760
2024 8 396
2024 9 249
2024 10 84
2024 11 128
2024 12 56
2025 1 176
2025 2 96
2025 3 256
2025 4 216
2025 5 136
2025 6 244
2025 7 192
2025 8 197
"""
orders = pd.read_csv(io.StringIO(orders_txt), sep=r"\s+")
orders["ds"] = pd.to_datetime(orders["SalesYear"].astype(str) + "-" +
                              orders["SalesMonth"].astype(str) + "-01")
df_orders = orders[["ds","TotalQty"]].rename(columns={"TotalQty":"orders"})

# ------------------ 3) Stok verisi ------------------
stocks_csv = """ItemCode,MonthStart,AyBasi_Stok
303-104092,2019-01-01,0
303-104092,2019-02-01,-22
303-104092,2019-03-01,-2
303-104092,2019-04-01,38
303-104092,2019-05-01,-7
303-104092,2019-06-01,-14
303-104092,2019-07-01,-28
303-104092,2019-08-01,4
303-104092,2019-09-01,68
303-104092,2019-10-01,1
303-104092,2019-11-01,105
303-104092,2019-12-01,84
303-104092,2020-01-01,18
303-104092,2020-02-01,54
303-104092,2020-03-01,70
303-104092,2020-04-01,349
303-104092,2020-05-01,356
303-104092,2020-06-01,192
303-104092,2020-07-01,166
303-104092,2020-08-01,262
303-104092,2020-09-01,181
303-104092,2020-10-01,165
303-104092,2020-11-01,1
303-104092,2020-12-01,133
303-104092,2021-01-01,117
303-104092,2021-02-01,32
303-104092,2021-03-01,31
303-104092,2021-04-01,111
303-104092,2021-05-01,128
303-104092,2021-06-01,-12
303-104092,2021-07-01,244
303-104092,2021-08-01,172
303-104092,2021-09-01,138
303-104092,2021-10-01,577
303-104092,2021-11-01,95
303-104092,2021-12-01,23
303-104092,2022-01-01,104
303-104092,2022-02-01,94
303-104092,2022-03-01,149
303-104092,2022-04-01,157
303-104092,2022-05-01,102
303-104092,2022-06-01,262
303-104092,2022-07-01,281
303-104092,2022-08-01,232
303-104092,2022-09-01,199
303-104092,2022-10-01,181
303-104092,2022-11-01,223
303-104092,2022-12-01,198
303-104092,2023-01-01,68
303-104092,2023-02-01,76
303-104092,2023-03-01,48
303-104092,2023-04-01,92
303-104092,2023-05-01,28
303-104092,2023-06-01,100
303-104092,2023-07-01,212
303-104092,2023-08-01,188
303-104092,2023-09-01,49
303-104092,2023-10-01,132
303-104092,2023-11-01,92
303-104092,2023-12-01,32
303-104092,2024-01-01,-8
303-104092,2024-02-01,104
303-104092,2024-03-01,-12
303-104092,2024-04-01,144
303-104092,2024-05-01,72
303-104092,2024-06-01,216
303-104092,2024-07-01,24
303-104092,2024-08-01,4
303-104092,2024-09-01,248
303-104092,2024-10-01,216
303-104092,2024-11-01,127
303-104092,2024-12-01,167
303-104092,2025-01-01,153
303-104092,2025-02-01,145
303-104092,2025-03-01,97
303-104092,2025-04-01,78
303-104092,2025-05-01,139
303-104092,2025-06-01,187
303-104092,2025-07-01,35
303-104092,2025-08-01,-29
303-104092,2025-09-01,130
"""
stocks = pd.read_csv(io.StringIO(stocks_csv))
stocks["ds"] = pd.to_datetime(stocks["MonthStart"])
df_stocks = stocks[["ds","AyBasi_Stok"]].rename(columns={"AyBasi_Stok":"stock"})

# ------------------ 4) Merge master & types ------------------
df = (
    df_sales
    .merge(df_orders, on="ds", how="left")
    .merge(df_stocks, on="ds", how="left")
    .sort_values("ds")
    .reset_index(drop=True)
)

for col in ["y", "orders", "stock"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# ------------------ 5) Calendar features ------------------
df["year"]    = df["ds"].dt.year
df["month"]   = df["ds"].dt.month
df["quarter"] = df["ds"].dt.quarter
df["season"]  = df["month"].apply(month_to_season)
season_dum = pd.get_dummies(df["season"], prefix="is", dtype=int)
df = pd.concat([df, season_dum], axis=1)

# ------------------ 6) Lags & ratio ------------------
df["y_lag1"]       = df["y"].shift(1)
df["y_lag12"]      = df["y"].shift(12)
df["orders_lag1"]  = df["orders"].shift(1)
df["orders_lag12"] = df["orders"].shift(12)
df["stock_lag1"]   = df["stock"].shift(1)
df["stock_lag12"]  = df["stock"].shift(12)
df["orders_ratio"] = df["orders"] / df["stock"].replace(0, np.nan)
df.replace([np.inf, -np.inf], np.nan, inplace=True)

# ------------------ 7) FX (EUR/TRY & USD/TRY via cross) ------------------
#  - EUR/TRY  : M.TRY.EUR.SP00.A  (TRY per EUR)
#  - EUR/USD  : M.USD.EUR.SP00.A  (USD per EUR)
#  - USD/TRY  = (EUR/TRY) / (EUR/USD)
try:
    eurtry_df = fetch_ecb_sdmx_csv("M.TRY.EUR.SP00.A", START_PERIOD, END_PERIOD).rename(columns={"value":"eurtry"})
    eurusd_df = fetch_ecb_sdmx_csv("M.USD.EUR.SP00.A", START_PERIOD, END_PERIOD).rename(columns={"value":"eurusd"})
    df = safe_merge(df, eurtry_df)
    df = safe_merge(df, eurusd_df)
    df["usdtry"] = df["eurtry"] / df["eurusd"]
    df[["eurtry","eurusd","usdtry"]] = df[["eurtry","eurusd","usdtry"]].ffill().bfill()
except Exception as e:
    print(f"[UYARI] ECB'den FX verisi alınamadı: {e}")
    print("        eurtry/usdtry sütunları NaN kalacaktır. İsterseniz yerel CSV ile doldurabilirsiniz.")
    df["eurtry"] = np.nan
    df["eurusd"] = np.nan
    df["usdtry"] = np.nan
    # Örnek (yerel CSV):
    # fx_local = pd.read_csv("fx_monthly_try.csv", parse_dates=["ds"])
    # df = safe_merge(df, fx_local[["ds","eurtry","eurusd","usdtry"]])

# ------------------ 8) Kontrol & Kaydet ------------------
print(df.head(96))
print("\nKolonlar:", df.columns.tolist())

# CSV'ye yazmak isterseniz:
df.to_csv("veri_matrisi_final_sales_orders_stock_calendar_lags_fx.csv", index=False)


           ds    y  orders  stock  year  month  quarter  season  is_Fall  \
0  2019-01-01   22      16      0  2019      1        1  Winter        0   
1  2019-02-01   60      64    -22  2019      2        1  Winter        0   
2  2019-03-01   60      72     -2  2019      3        1  Spring        0   
3  2019-04-01   45      37     38  2019      4        2  Spring        0   
4  2019-05-01  127     135     -7  2019      5        2  Spring        0   
..        ...  ...     ...    ...   ...    ...      ...     ...      ...   
74 2025-03-01  218     256     97  2025      3        1  Spring        0   
75 2025-04-01  219     216     78  2025      4        2  Spring        0   
76 2025-05-01  112     136    139  2025      5        2  Spring        0   
77 2025-06-01  244     244    187  2025      6        2  Summer        0   
78 2025-07-01   84     192     35  2025      7        3  Summer        0   

    is_Spring  ...  y_lag1  y_lag12  orders_lag1  orders_lag12  stock_lag1  \
0        